# Biological Explanation Evaluation (LLM-as-Judge)

Evaluates the quality of biological explanations generated by the Mistral-7B-Instruct RAG pipeline.

### Approach
1. Parse each numbered section in `biological_explanation` as an independent statement
2. Feed **all statements together** (in one prompt) plus the retrieved evidence to a **configurable judge LLM**
3. The judge assigns one score per statement using a **5-level rubric**:

| Score | AMP meaning | non-AMP meaning |
|-------|-------------|-----------------|
| **1.0** | Biologically plausible + directly supported by retrieved evidence | Well-grounded in sequence properties and retrieved UniProt context |
| **0.8** | Biologically plausible + partially supported by evidence | Partially supported by context — reasonable inference |
| **0.5** | Biologically plausible but speculative — not directly grounded in evidence | Logically consistent but not directly supported by context |
| **0.3** | Biologically questionable — partially incorrect or unsupported generalisation | Contains inaccuracies about the peptide's biological properties |
| **0.0** | Not biologically plausible — contradicts retrieved evidence | Contradicts known biology or cannot be inferred from sequence and context |

4. Per-explanation score = **mean of all statement scores** (0.0 values are included, not filtered)


| Backend | Model |
|---------|-------|
| `ollama` | `qwen2.5:7b` | 
| `anthropic` | `claude-haiku-4-5` | 


## 1. Configuration

Change these parameters before running.

In [1]:
# ═══════════════════════════════════════════════════════════════════════════
# CONFIGURATION — edit these before running
# ═══════════════════════════════════════════════════════════════════════════

# Judge LLM backend: "ollama" | "anthropic"
JUDGE_BACKEND  = "anthropic"

# Model name — must match backend:
#   ollama   : "qwen2.5:7b" (recommended) | "llama3.1:8b" | "gemma2:9b"
#   anthropic: "claude-haiku-4-5" (cost-effective) | "claude-sonnet-4-6" (highest accuracy)
JUDGE_MODEL    = "claude-haiku-4-5"

# Sampling temperature (0 = deterministic, reproducible)
TEMPERATURE    = 0

# ── Row selection (priority: SAMPLE_PCT > NUM_ROWS > all) ──────────────────
# Percentage of EACH dataset to evaluate independently:
#   SAMPLE_PCT = 100.0  → 10% of AMP rows  +  10% of non-AMP rows
#   SAMPLE_PCT = None  → falls back to NUM_ROWS (or all rows if NUM_ROWS is also None)
# Example: 358 AMP × 10% = 36 rows;  354 non-AMP × 10% = 36 rows = 72 total
SAMPLE_PCT = 100.0

# Fixed number of rows — used only when SAMPLE_PCT is None
NUM_ROWS = None

# Save a checkpoint CSV every N rows
CHECKPOINT_EVERY = 50

# ── Re-run control ─────────────────────────────────────────────────────────
# FORCE_RERUN = False → resume from checkpoint (safe default)
# FORCE_RERUN = True   → ignore checkpoint, re-evaluate all rows from scratch
#                        (use after changing prompts, model, or SAMPLE_PCT)
FORCE_RERUN = True

# Ollama server URL
OLLAMA_BASE_URL = "http://localhost:11434"

# Output paths
import os
from pathlib import Path

PROJECT_ROOT   = Path(os.getcwd()).parent
RESULTS_DIR    = PROJECT_ROOT / "results"
PLOTS_DIR      = PROJECT_ROOT / "plots" / "evaluation" / "bio_eval"
AMP_CSV        = RESULTS_DIR / "amp_rag_results.csv"
NON_AMP_CSV    = RESULTS_DIR / "non_amp_rag_results.csv"
CHECKPOINT_CSV = RESULTS_DIR / "bio_eval_checkpoint.csv"
OUTPUT_CSV     = RESULTS_DIR / "bio_eval_results.csv"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Judge backend    : {JUDGE_BACKEND}")
print(f"Judge model      : {JUDGE_MODEL}")
print(f"Temperature      : {TEMPERATURE}")
if SAMPLE_PCT is not None:
    print(f"Sample %         : {SAMPLE_PCT}%  (separate % applied to AMP and non-AMP)")
    print(f"  → ~{SAMPLE_PCT}% of 358 AMP  = ~{int(358*SAMPLE_PCT/100)} rows")
    print(f"  → ~{SAMPLE_PCT}% of 354 non-AMP = ~{int(354*SAMPLE_PCT/100)} rows")
else:
    print(f"Sample %         : None  → using NUM_ROWS={NUM_ROWS} (or ALL if None)")
print(f"Checkpoint every : {CHECKPOINT_EVERY}")
print(f"Force rerun      : {FORCE_RERUN}  {'← checkpoint will be ignored' if FORCE_RERUN else '← resumes from checkpoint if it exists'}")
print(f"Output CSV       : {OUTPUT_CSV}")


## 2. Imports & Environment

In [2]:
!pip install python-dotenv -q

In [3]:
import os, json, re, time, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import requests
from tqdm.auto import tqdm

try:
    from dotenv import load_dotenv
    loaded = load_dotenv(PROJECT_ROOT / ".env")
    if not loaded:
        print("Warning: .env file not found or empty")
except ImportError:
    print("Warning: python-dotenv not installed — run: !pip install python-dotenv -q")

warnings.filterwarnings("ignore")

# ── Global plot style (all text bold) ─────────────────────────────────────────
plt.rcParams.update({
    "font.weight":       "bold",
    "axes.labelweight":  "bold",
    "axes.titleweight":  "bold",
    "axes.titlesize":    14,
    "axes.labelsize":    12,
    "xtick.labelsize":   11,
    "ytick.labelsize":   11,
    "legend.fontsize":   11,
    "figure.dpi":        120,
    "figure.facecolor":  "white",
})
sns.set_theme(style="whitegrid")

print("Imports OK")
print(f"ANTHROPIC_API_KEY set: {'Yes' if os.getenv('ANTHROPIC_API_KEY') else 'No (needed only for anthropic backend)'}")


## 3. Backend Setup

### Anthropic backend only
If `JUDGE_BACKEND = "anthropic"`, uncomment and run the cell below first:


In [4]:
# Uncomment if using Anthropic backend
!pip install anthropic -q


In [5]:
# ── Ollama health check (runs only if JUDGE_BACKEND == "ollama") ─────────────
if JUDGE_BACKEND == "ollama":
    try:
        r = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
        r.raise_for_status()
        available = [m["name"] for m in r.json().get("models", [])]
        print(f"Ollama server  : OK")
        print(f"Models available: {available}")
        if JUDGE_MODEL in available:
            print(f"Judge model    : {JUDGE_MODEL} — ready")
        else:
            print(f"Judge model    : {JUDGE_MODEL} — NOT FOUND")
            print(f"  Run this to download it:")
            print(f"  !ollama pull {JUDGE_MODEL}")
    except requests.exceptions.ConnectionError:
        print("Ollama not reachable — start with: ollama serve")
    except Exception as e:
        print(f"Health check error: {e}")
else:
    api_key = os.getenv("ANTHROPIC_API_KEY", "")
    if api_key and api_key != "your_anthropic_api_key_here":
        print(f"Anthropic API key: set ({api_key[:8]}...)")
    else:
        print("Anthropic API key: MISSING — set ANTHROPIC_API_KEY in .env")


## 4. Load Data

In [10]:
df_amp     = pd.read_csv(AMP_CSV)
df_non_amp = pd.read_csv(NON_AMP_CSV)

df_amp["dataset"]     = "AMP"
df_non_amp["dataset"] = "non-AMP"

df_all = pd.concat([df_amp, df_non_amp], ignore_index=True)

print(f"AMP predictions    : {len(df_amp)}  rows  (TP={sum(df_amp.prediction_result=='TP')}, FP={sum(df_amp.prediction_result=='FP')})")
print(f"non-AMP predictions: {len(df_non_amp)}  rows  (TN={sum(df_non_amp.prediction_result=='TN')}, FN={sum(df_non_amp.prediction_result=='FN')})")
print(f"Total              : {len(df_all)}  rows")
print(f"\nColumns: {list(df_all.columns)}")


## 5. Statement Extraction

Each `biological_explanation` contains numbered sections (e.g. `1. Title: body text...`).
This function splits them into individual statements for per-statement scoring.


In [9]:
def extract_statements(explanation: str) -> list[dict]:
    """
    Split a numbered biological explanation into individual statements.

    Handles both formats:
      '1. Title: body'    (AMP explanations)
      '1. TITLE: body'    (non-AMP explanations)

    Returns list of dicts: {num, title, body}
    """
    if not isinstance(explanation, str) or not explanation.strip():
        return []

    # Split on newline followed by a digit and period
    chunks = re.split(r'\n(?=\d+\.)', '\n' + explanation.strip())
    statements = []
    for chunk in chunks:
        chunk = chunk.strip()
        if not chunk:
            continue
        # Match "N. Title: body" or "N. Title\nbody"
        m = re.match(r'^(\d+)\.\s+([^:\n]+)[:\n]\s*(.*)', chunk, re.DOTALL)
        if m:
            statements.append({
                "num":   int(m.group(1)),
                "title": m.group(2).strip(),
                "body":  m.group(3).strip(),
            })
        else:
            # Fallback: treat whole chunk as body
            num_m = re.match(r'^(\d+)\.\s*(.*)', chunk, re.DOTALL)
            if num_m:
                statements.append({
                    "num":   int(num_m.group(1)),
                    "title": "Statement",
                    "body":  num_m.group(2).strip(),
                })
    return statements


# ── Smoke test ────────────────────────────────────────────────────────────────
stmts = extract_statements(df_all["biological_explanation"].iloc[0])
print(f"Statements found in row 0: {len(stmts)}")
for s in stmts:
    print(f"  [{s['num']}] {s['title']}: {s['body'][:80]}...")


## 6. Evidence Builder

Formats the top-3 retrieved hits into a readable evidence block for the judge.

In [9]:
def fmt(pipe_str, max_items=6):
    """Convert pipe-separated string to comma list, handle NaN."""
    if not isinstance(pipe_str, str): return "Not specified"
    items = [x.strip() for x in pipe_str.split("|") if x.strip()][:max_items]
    return ", ".join(items) if items else "Not specified"


def build_evidence(row, label=None) -> str:
    """Build a labelled evidence block from the three retrieved hits.
    Uses StarPep columns for AMP rows, UniProt Swiss-Prot columns for non-AMP rows."""
    is_amp = row.get("dataset", "AMP") == "AMP"
    if label is None:
        label = (
            "Retrieved Evidence (top-3 cosine-similar AMPs from StarPep)"
            if is_amp else
            "Retrieved UniProt Swiss-Prot Neighbours (comparison context only — not required grounding)"
        )
    lines = [f"=== {label} ==="]
    for n in [1, 2, 3]:
        sim = row.get(f"hit{n}_cosine_sim", "N/A")
        sim_str = f"{sim:.3f}" if isinstance(sim, (int, float)) else str(sim)
        if is_amp:
            sid  = row.get(f"hit{n}_starpep_id", "N/A")
            seq  = row.get(f"hit{n}_sequence",   "N/A")
            act  = fmt(row.get(f"hit{n}_activities",    ""))
            tgt  = fmt(row.get(f"hit{n}_targets",       ""), 8)
            src  = fmt(row.get(f"hit{n}_source",        ""))
            mods = fmt(row.get(f"hit{n}_modifications", ""))
            lines.append(
                f"Hit {n} | {sid} | cosine_sim={sim_str}\n"
                f"  Sequence   : {seq}\n"
                f"  Activities : {act}\n"
                f"  Targets    : {tgt}\n"
                f"  Source     : {src}\n"
                f"  Mods       : {mods}"
            )
        else:
            uid  = row.get(f"hit{n}_uniprot_id",   "N/A")
            seq  = row.get(f"hit{n}_sequence",      "N/A")
            name = row.get(f"hit{n}_protein_name",  "Not specified")
            kw   = fmt(row.get(f"hit{n}_keywords",  ""))
            org  = row.get(f"hit{n}_organism",      "Not specified")
            fn   = row.get(f"hit{n}_function",      "Not specified")
            lines.append(
                f"Hit {n} | {uid} | cosine_sim={sim_str}\n"
                f"  Sequence     : {seq}\n"
                f"  Protein name : {name}\n"
                f"  Keywords     : {kw}\n"
                f"  Organism     : {org}\n"
                f"  Function     : {fn}"
            )
    return "\n".join(lines)


## 7. Judge Prompts

Two separate system prompts are used depending on whether the peptide is AMP or non-AMP.
**Both** receive the retrieved evidence block so the judge can assess grounding.

### AMP prompt — biological plausibility + evidence grounding

| Score | Meaning |
|-------|---------|
| **1.0** | The claim is biologically plausible and **supports** the retrieved evidence (activities, targets, or source organisms match) |
| **0.8** | The claim is biologically plausible and **partially supports** the retrieved evidence — reasonable inference from the evidence |
| **0.5** | The claim is biologically plausible and **speculative or inferred** — logically consistent but not directly grounded in the evidence |
| **0.3** | The claim is **biologically questionable** — partially incorrect or unsupported generalisation not grounded in the evidence |
| **0.0** | The claim is **not biologically plausible** and not grounded in the retrieved evidence — contradicts known AMP biology or the retrieved evidence |

### non-AMP prompt — biological plausibility of absence reasoning

| Score | Meaning |
|-------|---------|
| **1.0** | Well-grounded — the explanation of the peptide's biological properties is accurate and supported by the retrieved UniProt context |
| **0.8** | Partially plausible — broadly correct but may oversimplify or make inferences beyond what the context directly supports |
| **0.5** | Speculative — logically consistent but not directly supported by the retrieved context |
| **0.3** | Biologically questionable — contains inaccuracies or unsupported generalisations about the peptide's properties |
| **0.0** | Not biologically plausible — contradicts known biology or cannot be inferred from the sequence and context |


In [8]:
# ── AMP system prompt: biological plausibility + evidence grounding ───────
AMP_SYSTEM_PROMPT = """You are a rigorous bioinformatics expert evaluating the biological plausibility of RAG-generated antimicrobial peptide (AMP) explanations.

You will be given:
1. A biological explanation split into clearly numbered statements
2. The retrieved evidence (top-3 cosine-similar AMPs from the StarPep database) used to generate it

For EACH numbered statement, assign exactly ONE of these scores:
  1.0 = The claim is biologically plausible and supports the retrieved evidence (e.g., mentioned activities, targets, or source organisms match)
  0.8 = The claim is biologically plausible and partially supports the retrieved evidence — biologically plausible or reasonable inference from the evidence
  0.5 = The claim is biologically plausible and speculative or inferred — logically consistent but not directly grounded in the retrieved evidence
  0.3 = The claim is biologically questionable — partially incorrect or contains unsupported generalisations not grounded in the retrieved evidence
  0.0 = The claim is not biologically plausible and not grounded in the retrieved evidence — contradicts known AMP biology or the retrieved evidence

Rules:
- Score EVERY statement — do not skip any
- A statement that contradicts or ignores the retrieved evidence should receive 0.0
- Return ONLY valid JSON in this exact format with no other text:

{
  "statements": [
    {"num": 1, "title": "Section Title", "score": 1.0, "reason": "one sentence"},
    {"num": 2, "title": "Section Title", "score": 0.8, "reason": "one sentence"}
  ]
}
"""

# ── non-AMP system prompt ─────────────────────────────────────────────────
NON_AMP_SYSTEM_PROMPT = """You are a rigorous bioinformatics expert evaluating the biological plausibility of explanations generated by a Mistral 7B model for a peptide which is possibly not an antimicrobial peptide.

You will be given:
1. A biological explanation split into clearly numbered statements
2. A set of peptides retrieved from the UniProt Swiss-Prot database based on cosine similarity in protein embedding space — these are the closest structural neighbours of the query peptide in the database, provided as comparison context only, not as required grounding

For EACH numbered statement, assess biological plausibility and assign scores as below:
  1.0 = biologically plausible and well-grounded in the sequence properties and retrieved context
  0.8 = biologically plausible and partially supported by the retrieved context — reasonable inference
  0.5 = biologically plausible but speculative — logically consistent but not directly supported by the retrieved context
  0.3 = biologically questionable — contains inaccuracies or unsupported generalisations about the peptide's properties
  0.0 = not biologically plausible — contradicts known biology or cannot be inferred from the sequence and context

Rules:
- Score EVERY statement — do not skip any
- Return ONLY valid JSON in this exact format with no other text:

{
  "statements": [
    {"num": 1, "title": "Section Title", "score": 1.0, "reason": "one sentence"},
    {"num": 2, "title": "Section Title", "score": 0.8, "reason": "one sentence"}
  ]
}
"""


def build_user_prompt(row) -> str:
    """
    Build user prompt with all statements in one block.
    AMP rows: include hydrophobicity and net charge; evidence is required grounding.
    non-AMP rows: length only; retrieved peptides are context only, not required grounding.
    """
    stmts = extract_statements(row["biological_explanation"])
    if stmts:
        stmt_block = "\n\n".join(
            f"Statement {s['num']} — {s['title']}:\n{s['body']}"
            for s in stmts
        )
    else:
        stmt_block = row["biological_explanation"]

    if row["dataset"] == "AMP":
        query_info = (
            f"Query peptide: {row['query_sequence']}\n"
            f"AMP probability: {row['prob_amp']:.3f}\n"
            f"Hydrophobicity: {row['hydrophobicity_pct']:.1f}%  |  "
            f"Net charge: {row['net_charge']}  |  "
            f"Length: {row['query_length']}"
        )
        evidence = build_evidence(row)
        return (
            f"{query_info}\n\n"
            f"{evidence}\n\n"
            f"=== Statements to Evaluate ===\n\n"
            f"{stmt_block}\n\n"
            f"Score each statement using the 5-level rubric. Return ONLY the JSON."
        )
    else:
        query_info = (
            f"Query peptide: {row['query_sequence']}\n"
            f"AMP probability: {row['prob_amp']:.3f}\n"
            f"Length: {row['query_length']}"
        )
        evidence = build_evidence(row)
        return (
            f"{query_info}\n\n"
            f"{evidence}\n\n"
            f"=== Statements to Evaluate (non-AMP reasoning) ===\n\n"
            f"{stmt_block}\n\n"
            f"Score each statement on the biological plausibility of the explanation. "
            f"Return ONLY the JSON."
        )


print("AMP prompt length    :", len(AMP_SYSTEM_PROMPT))
print("non-AMP prompt length:", len(NON_AMP_SYSTEM_PROMPT))
print("User prompt (AMP row):", len(build_user_prompt(df_all[df_all['dataset']=='AMP'].iloc[0])))
print("User prompt (non-AMP):", len(build_user_prompt(df_all[df_all['dataset']=='non-AMP'].iloc[0])))
print()
print("Score levels: 1.0 / 0.8 / 0.5 / 0.3 / 0.0")
print("AMP rows     → AMP_SYSTEM_PROMPT (evidence grounding + biological plausibility)")
print("non-AMP rows → NON_AMP_SYSTEM_PROMPT (biological plausibility of peptide properties)")

## 8. LLM Call Functions

Two implementations behind a common `call_judge()` interface.

In [7]:
def call_ollama(user_prompt: str, system_prompt: str, retries: int = 3) -> str:
    """Call Ollama chat API. Returns raw LLM response string."""
    for attempt in range(retries):
        try:
            r = requests.post(
                f"{OLLAMA_BASE_URL}/api/chat",
                json={
                    "model":    JUDGE_MODEL,
                    "messages": [
                        {"role": "system", "content": system_prompt},
                        {"role": "user",   "content": user_prompt},
                    ],
                    "stream":  False,
                    "format":  "json",
                    "options": {"temperature": TEMPERATURE, "num_predict": 1500},
                },
                timeout=180,
            )
            r.raise_for_status()
            return r.json()["message"]["content"].strip()
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
            else:
                raise RuntimeError(f"Ollama call failed after {retries} attempts: {e}")


def call_anthropic_api(user_prompt: str, system_prompt: str, retries: int = 3) -> str:
    """Call Anthropic Messages API. Returns raw LLM response string."""
    try:
        import anthropic
    except ImportError:
        raise ImportError("Run: !pip install anthropic -q  then restart kernel")

    # Defensive re-load in case kernel was started before python-dotenv was installed
    try:
        from dotenv import load_dotenv
        load_dotenv(PROJECT_ROOT / ".env")
    except ImportError:
        pass

    api_key = os.getenv("ANTHROPIC_API_KEY", "")
    if not api_key or api_key == "your_anthropic_api_key_here":
        raise ValueError("ANTHROPIC_API_KEY not set — add it to .env and restart kernel")

    client = anthropic.Anthropic(api_key=api_key)
    for attempt in range(retries):
        try:
            msg = client.messages.create(
                model=JUDGE_MODEL,
                max_tokens=1500,
                temperature=TEMPERATURE,
                system=system_prompt,
                messages=[{"role": "user", "content": user_prompt}],
            )
            return msg.content[0].text.strip()
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
            else:
                raise RuntimeError(f"Anthropic call failed after {retries} attempts: {e}")


def call_judge(user_prompt: str, system_prompt: str) -> str:
    """Unified interface — routes to configured backend."""
    if JUDGE_BACKEND == "ollama":
        return call_ollama(user_prompt, system_prompt)
    elif JUDGE_BACKEND == "anthropic":
        return call_anthropic_api(user_prompt, system_prompt)
    else:
        raise ValueError(f"Unknown JUDGE_BACKEND: {JUDGE_BACKEND!r}")


def parse_judge_response(raw: str) -> list[dict]:
    """Extract JSON from judge response; return list of statement dicts."""
    raw = re.sub(r'^```(?:json)?\s*', '', raw.strip(), flags=re.MULTILINE)
    raw = re.sub(r'```$', '', raw.strip(), flags=re.MULTILINE)
    data = json.loads(raw)
    return data.get("statements", [])


VALID_SCORES = {1.0, 0.8, 0.5, 0.3, 0.0}


def score_row(row) -> dict:
    """
    Score a single explanation row using the 5-level rubric.
    Selects AMP_SYSTEM_PROMPT for AMP rows, NON_AMP_SYSTEM_PROMPT for non-AMP rows.
    Returns dict with mean_score, num_statements, statement_scores_json, parse_error.
    """
    system_prompt = AMP_SYSTEM_PROMPT if row["dataset"] == "AMP" else NON_AMP_SYSTEM_PROMPT
    user_prompt   = build_user_prompt(row)
    raw = call_judge(user_prompt, system_prompt)
    try:
        statements = parse_judge_response(raw)
        scores = [float(s["score"]) for s in statements if "score" in s]
        if not scores:
            raise ValueError("No scores parsed")
        # Clamp any unexpected values to nearest valid score
        scores = [min(VALID_SCORES, key=lambda v: abs(v - s)) for s in scores]
        return {
            "mean_score":            round(sum(scores) / len(scores), 4),
            "num_statements":        len(statements),
            "statement_scores_json": json.dumps(statements),
            "parse_error":           False,
        }
    except Exception as e:
        # Fallback: extract raw score values from JSON text
        found = re.findall(r'"score"\s*:\s*(1\.0|0\.8|0\.5|0\.3|0(?:\.0)?)', raw)
        scores = [float(x) for x in found]
        mean_score = round(sum(scores) / len(scores), 4) if scores else 0.0
        return {
            "mean_score":            mean_score,
            "num_statements":        len(scores) if scores else 0,
            "statement_scores_json": json.dumps({"raw": raw[:500], "error": str(e)}),
            "parse_error":           True,
        }

print("Judge functions defined. Backend:", JUDGE_BACKEND, "| Model:", JUDGE_MODEL)
print("Score levels: 1.0 / 0.8 / 0.5 / 0.3 / 0.0")
print("AMP rows     → AMP_SYSTEM_PROMPT (biological plausibility + evidence)")
print("non-AMP rows → NON_AMP_SYSTEM_PROMPT (biological plausibility)")


In [6]:
# ── Print example prompts sent to the judge ───────────────────────────────
SEP = "=" * 80

# ── AMP example ───────────────────────────────────────────────────────────
amp_row = df_all[df_all["dataset"] == "AMP"].iloc[0]
amp_user = build_user_prompt(amp_row)

print(SEP)
print("SYSTEM PROMPT  (AMP)")
print(SEP)
print(AMP_SYSTEM_PROMPT)

print(SEP)
print("USER PROMPT  (AMP — first row)")
print(SEP)
print(amp_user)

# ── non-AMP example ───────────────────────────────────────────────────────
non_amp_row = df_all[df_all["dataset"] == "non-AMP"].iloc[0]
non_amp_user = build_user_prompt(non_amp_row)

print()
print(SEP)
print("SYSTEM PROMPT  (non-AMP)")
print(SEP)
print(NON_AMP_SYSTEM_PROMPT)

print(SEP)
print("USER PROMPT  (non-AMP — first row)")
print(SEP)
print(non_amp_user)


## 9. Smoke Test — Score First Row

Runs the full pipeline on a single row. Check output before running the full loop.

In [11]:
print("=== Scoring row 0 ===")
t0 = time.time()
result = score_row(df_all.iloc[0])
elapsed = time.time() - t0

print(f"Mean score    : {result['mean_score']}")
print(f"Statements    : {result['num_statements']}")
print(f"Parse error   : {result['parse_error']}")
print(f"Elapsed       : {elapsed:.1f}s")
print()

# Pretty-print per-statement scores
stmts = json.loads(result["statement_scores_json"])
if isinstance(stmts, list):
    print(f"{'#':<4} {'Score':<7} {'Title':<35} Reason")
    print("-" * 90)
    for s in stmts:
        print(f"{s.get('num','?'):<4} {s.get('score','?'):<7} {str(s.get('title',''))[:34]:<35} {str(s.get('reason',''))[:60]}")


## 10. Evaluation Loop

- Resumes from `bio_eval_checkpoint.csv` if it exists (safe to re-run)
- Saves checkpoint every `CHECKPOINT_EVERY` rows
- Shows progress with tqdm

> Set `NUM_ROWS = None` in Cell 2 and re-run from the top to evaluate all 712 rows.


In [12]:
import numpy as np

# ── Determine rows to process ─────────────────────────────────────────────
if SAMPLE_PCT is not None:
    n_amp = max(1, int(np.ceil(len(df_amp) * SAMPLE_PCT / 100)))
    n_non = max(1, int(np.ceil(len(df_non_amp) * SAMPLE_PCT / 100)))
    df_eval = pd.concat([df_amp.head(n_amp), df_non_amp.head(n_non)], ignore_index=True)
    print(f"Sampling {SAMPLE_PCT}%: {n_amp} AMP + {n_non} non-AMP = {len(df_eval)} rows")
elif NUM_ROWS:
    df_eval = df_all.head(NUM_ROWS).copy()
    print(f"Fixed NUM_ROWS={NUM_ROWS}: {len(df_eval)} rows")
else:
    df_eval = df_all.copy()
    print(f"Full dataset: {len(df_eval)} rows")

# ── Resume from checkpoint or start fresh ────────────────────────────────
already_done   = set()
checkpoint_rows = []

if FORCE_RERUN:
    print("FORCE_RERUN=True — ignoring checkpoint, evaluating all rows from scratch")
elif CHECKPOINT_CSV.exists():
    df_ckpt = pd.read_csv(CHECKPOINT_CSV)
    already_done = set(df_ckpt["seq_idx"].astype(str).tolist())
    checkpoint_rows = df_ckpt.to_dict("records")
    print(f"Resuming — {len(already_done)} rows already done (set FORCE_RERUN=True to re-evaluate)")

pending = df_eval[~df_eval["seq_idx"].astype(str).isin(already_done)]
print(f"Rows to evaluate: {len(pending)}  (total in eval set: {len(df_eval)})")


In [13]:
# ── Main evaluation loop ─────────────────────────────────────────────────────
eval_rows = checkpoint_rows.copy()
errors    = 0

for i, (_, row) in enumerate(tqdm(pending.iterrows(), total=len(pending), desc="Scoring")):
    result = score_row(row)
    if result["parse_error"]:
        errors += 1

    is_amp = row["dataset"] == "AMP"
    eval_rows.append({
        "seq_idx":               row["seq_idx"],
        "query_sequence":        row["query_sequence"],
        "dataset":               row["dataset"],
        "prediction_result":     row["prediction_result"],
        "prob_amp":              row["prob_amp"],
        "true_label":            row["true_label"],
        "query_length":          row["query_length"],
        "hydrophobicity_pct":    row.get("hydrophobicity_pct", None) if is_amp else None,
        "net_charge":            row.get("net_charge", None) if is_amp else None,
        "hit1_cosine_sim":       row["hit1_cosine_sim"],
        "hit2_cosine_sim":       row["hit2_cosine_sim"],
        "hit3_cosine_sim":       row["hit3_cosine_sim"],
        "mean_score":            result["mean_score"],
        "num_statements":        result["num_statements"],
        "statement_scores_json": result["statement_scores_json"],
        "parse_error":           result["parse_error"],
        "judge_backend":         JUDGE_BACKEND,
        "judge_model":           JUDGE_MODEL,
    })
    # Checkpoint
    if (i + 1) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(eval_rows).to_csv(CHECKPOINT_CSV, index=False)
        tqdm.write(f"Checkpoint saved — {len(eval_rows)} rows done, {errors} errors")

# Final save
df_results = pd.DataFrame(eval_rows)
df_results.to_csv(CHECKPOINT_CSV, index=False)
df_results.to_csv(OUTPUT_CSV, index=False)
print(f"\nDone. {len(df_results)} rows evaluated, {errors} parse errors.")
print(f"Saved to: {OUTPUT_CSV}")

## 11. Summary Statistics

In [14]:
df_r = pd.read_csv(OUTPUT_CSV)
print(f"Loaded {len(df_r)} rows from {OUTPUT_CSV.name}")
print(f"Judge: {df_r['judge_model'].iloc[0]}  ({df_r['judge_backend'].iloc[0]})")
print()

# ── By dataset ───────────────────────────────────────────────────────────────
print("=== Score by Dataset ===")
grp1 = df_r.groupby("dataset")["mean_score"].agg(["mean","median","std","count"])
grp1.columns = ["Mean","Median","Std","N"]
print(grp1.round(4).to_string())
print()

# ── By prediction type ────────────────────────────────────────────────────────
print("=== Score by Prediction Type ===")
grp2 = df_r.groupby("prediction_result")["mean_score"].agg(["mean","median","std","count"])
grp2.columns = ["Mean","Median","Std","N"]
print(grp2.round(4).to_string())
print()

# ── Parse errors ─────────────────────────────────────────────────────────────
n_err = df_r["parse_error"].sum()
print(f"Parse errors: {n_err}/{len(df_r)} ({n_err/len(df_r)*100:.1f}%)")
print()

# ── Overall ───────────────────────────────────────────────────────────────────
print(f"Overall mean score : {df_r['mean_score'].mean():.4f}")
print(f"Overall median     : {df_r['mean_score'].median():.4f}")
print(f"Overall std        : {df_r['mean_score'].std():.4f}")
print(f"Score range        : [{df_r['mean_score'].min():.3f}, {df_r['mean_score'].max():.3f}]")
